# Preprocessing and validation folds


In [8]:
from pathlib import Path

import numpy as np
import polars as pl
from sklearn.model_selection import StratifiedKFold

In [18]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
processed_data_dir.mkdir(parents=True, exist_ok=True)

train_path = raw_data_dir / "train.csv"
test_path = raw_data_dir / "test.csv"

train_df = pl.read_csv(train_path)
test_df = pl.read_csv(test_path)

train_df.shape, test_df.shape

((699635, 23), (299844, 22))

In [4]:
binary_columns = {
    "Gender": {"Female": 0, "Male": 1},
    "Customer Type": {"disloyal Customer": 0, "Loyal Customer": 1},
    "Type of Travel": {"Personal Travel": 0, "Business travel": 1},
}

binary_columns

{'Gender': {'Female': 0, 'Male': 1},
 'Customer Type': {'disloyal Customer': 0, 'Loyal Customer': 1},
 'Type of Travel': {'Personal Travel': 0, 'Business travel': 1}}

In [5]:
train_df = train_df.with_columns(
    pl.col("satisfaction").cast(pl.Int8)
)

for column, mapping in binary_columns.items():
    train_df = train_df.with_columns(
        pl.col(column).replace_strict(mapping, return_dtype=pl.Int8)
    )
    test_df = test_df.with_columns(
        pl.col(column).replace_strict(mapping, return_dtype=pl.Int8)
    )

train_df.select([*binary_columns, "satisfaction"]).head()

Gender,Customer Type,Type of Travel,satisfaction
i8,i8,i8,i8
0,1,0,0
1,1,1,1
0,1,0,0
1,1,0,0
0,0,1,0


In [15]:
train_df = train_df.with_columns(
    pl.col("Arrival Delay in Minutes")
    .is_null()
    .cast(pl.Int8)
    .alias("Arrival Delay Missing")
)

test_df = test_df.with_columns(
    pl.col("Arrival Delay in Minutes")
    .is_null()
    .cast(pl.Int8)
    .alias("Arrival Delay Missing")
)

In [16]:
features = train_df.drop(["id", "satisfaction"])
target = train_df["satisfaction"]
fold_numbers = np.zeros(train_df.height, dtype=np.int8)

folds = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

for fold, (train_index, valid_index) in enumerate(
    folds.split(features, target)
):
    fold_numbers[valid_index] = fold

train_df = train_df.with_columns(
    pl.Series("fold", fold_numbers)
)

train_df.group_by("fold").len().sort("fold")

fold,len
i8,u32
0,139927
1,139927
2,139927
3,139927
4,139927


In [19]:
train_columns = set(train_df.columns) - {"satisfaction", "fold"}
test_columns = set(test_df.columns)

assert train_columns == test_columns
assert train_df["id"].n_unique() == train_df.height
assert test_df["id"].n_unique() == test_df.height

train_output_path = processed_data_dir / "train_with_folds.parquet"
test_output_path = processed_data_dir / "test_clean.parquet"

train_df.write_parquet(train_output_path)
test_df.write_parquet(test_output_path)

train_df.shape, test_df.shape

((699635, 25), (299844, 23))